In [5]:
pip install torch transformers pandas numpy scikit-learn matplotlib seaborn plotly wordcloud nltk textblob textstat optuna focal-loss torchcontrib tqdm adversarialbox


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement adversarialbox (from versions: none)
ERROR: No matching distribution found for adversarialbox


In [ ]:
# Simplified but still powerful training notebook
# With fallbacks for missing packages

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import random

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Configuration
class Config:
    model_name = 'bert-base-uncased'
    num_classes = 2
    max_length = 256
    batch_size = 16  # Reduced for compatibility
    epochs = 3  # Reduced for quick testing
    learning_rate = 2e-5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = Config()
print(f"Using device: {cfg.device}")

# Dataset class
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Load data
print("Loading data...")
try:
    fake_df = pd.read_csv('Data/fake and real news/Fake.csv')
    true_df = pd.read_csv('Data/fake and real news/True.csv')
    
    texts = pd.concat([fake_df['text'], true_df['text']], ignore_index=True)
    labels = np.array([0] * len(fake_df) + [1] * len(true_df))
    
    print(f"✓ Loaded {len(texts)} samples")
    print(f"  Fake: {sum(labels==0)}, Real: {sum(labels==1)}")
    
except FileNotFoundError:
    print("Dataset not found. Creating sample data...")
    # Create sample data
    np.random.seed(42)
    n_samples = 1000
    texts = [f"Sample news article {i} with some content" for i in range(n_samples)]
    labels = np.random.randint(0, 2, n_samples)
    print(f"✓ Created {n_samples} sample articles")

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train: {len(train_texts)}, Validation: {len(val_texts)}")

# Create data loaders
tokenizer = BertTokenizer.from_pretrained(cfg.model_name)
train_dataset = FakeNewsDataset(train_texts, train_labels, tokenizer, cfg.max_length)
val_dataset = FakeNewsDataset(val_texts, val_labels, tokenizer, cfg.max_length)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size)

# Create model
print("\nCreating model...")
model = BertForSequenceClassification.from_pretrained(
    cfg.model_name, 
    num_labels=cfg.num_classes
)
model = model.to(cfg.device)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=cfg.learning_rate)
total_steps = len(train_loader) * cfg.epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Training loop
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50)

train_losses = []
val_accuracies = []

for epoch in range(cfg.epochs):
    print(f"\nEpoch {epoch+1}/{cfg.epochs}")
    print("-" * 30)
    
    # Training
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch['input_ids'].to(cfg.device)
        attention_mask = batch['attention_mask'].to(cfg.device)
        labels = batch['labels'].to(cfg.device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch['input_ids'].to(cfg.device)
            attention_mask = batch['attention_mask'].to(cfg.device)
            labels = batch['labels'].to(cfg.device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    val_accuracies.append(accuracy)
    
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.4f}")

# Final evaluation
print("\n" + "="*50)
print("FINAL EVALUATION")
print("="*50)

# Get all predictions
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Final Evaluation"):
        input_ids = batch['input_ids'].to(cfg.device)
        attention_mask = batch['attention_mask'].to(cfg.device)
        labels = batch['labels'].to(cfg.device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(outputs.logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
conf_matrix = confusion_matrix(all_labels, all_preds)

print(f"\nFinal Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {conf_matrix[0,0]:6d} | False Positives: {conf_matrix[0,1]:6d}")
print(f"  False Negatives: {conf_matrix[1,0]:6d} | True Positives:  {conf_matrix[1,1]:6d}")

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Fake', 'Real']))

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

ax2.plot(val_accuracies, marker='o', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

# Save model
os.makedirs('../model', exist_ok=True)
model_save_path = '../model/bert_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': cfg.__dict__,
    'accuracy': accuracy,
    'f1_score': f1
}, model_save_path)
print(f"\n✓ Model saved to {model_save_path}")

# Save tokenizer
tokenizer.save_pretrained('../model/')
print("✓ Tokenizer saved")

print("\n" + "="*50)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("="*50)